In [26]:
!pip install -q \
langchain-community \
langchain-text-splitters \
transformers \
sentence-transformers \
faiss-cpu \
pypdf \
torch


In [27]:
from google.colab import files

uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]

print("Uploaded file:", PDF_PATH)


Saving syllabus.pdf to syllabus (1).pdf
Uploaded file: syllabus (1).pdf


In [28]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

print("Total pages loaded:", len(documents))


Total pages loaded: 2


In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)
print("Text chunks created:", len(chunks))


Text chunks created: 9


In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=30,
    separators=[
        "\n\n",
        "\nUNIT",
        "\nUnit",
        "\nCourse Outcomes",
        "\nObjectives",
        "\nCO",
        "\n",
        "."
    ]
)

chunks = splitter.split_documents(documents)

# Remove very small / noisy chunks
chunks = [c for c in chunks if len(c.page_content.strip()) > 50]

print("Chunks created:", len(chunks))


Chunks created: 19


In [30]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded")


Embeddings model loaded


In [31]:
from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(chunks, embeddings)

print("FAISS vector database created")


FAISS vector database created


In [32]:
from transformers import pipeline

llm = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_length=256,
    temperature=0.0,
    do_sample=False
)

print("Hugging Face LLM loaded")


Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Hugging Face LLM loaded


In [33]:
def retrieve_syllabus(question):
    keywords = [
        "outcome", "outcomes", "objective", "objectives",
        "unit", "course", "co", "aim", "syllabus"
    ]

    docs = vector_db.similarity_search(question, k=8)

    filtered_docs = []
    for doc in docs:
        text = doc.page_content.lower()
        if any(k in text for k in keywords):
            filtered_docs.append(doc)

    # Fallback if filtering removes everything
    return filtered_docs if filtered_docs else docs[:3]


In [34]:
def ask_syllabus(question):
    docs = retrieve_syllabus(question)

    context = "\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are a syllabus-only academic assistant.

STRICT RULES:
- Answer ONLY using the syllabus text below
- If the answer is NOT clearly mentioned, reply exactly:
  "This information is not specified in the syllabus."
- Do NOT guess, infer, or add outside knowledge

SYLLABUS TEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    response = llm(prompt)
    return response[0]["generated_text"]

print("Accurate RAG system initialized")


Accurate RAG system initialized


In [35]:
question = input("Enter your syllabus question: ")

answer = ask_syllabus(question)

print("\nQuestion:", question)
print("Answer:", answer)


Enter your syllabus question: text books

Question: text books
Answer: 1. Ramesh Sharda, Dursun Delen, Efraim Turban, Business Intelligence and Analytics, Pearson 10th edition, 2018
